# Symmetric Parseval CNN — denoising experiments (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Theborna/symmetric_parseval_conv/blob/main/colab_experiments.ipynb)

This notebook trains the four core models (`baseline`, `symmetric`, `mirror`,
`symmetric_mirror`) at several noise levels and produces the paper's PSNR/SSIM
table (Markdown + LaTeX).

**Before you start:** enable a GPU via *Runtime → Change runtime type → GPU*,
and have your BSD500 `train.h5` / `test.h5` ready (see step 4 for ways to get
them onto Colab — Drive is only one option).


## 1. Check the runtime


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch:', torch.__version__, '| device:', device)
if device == 'cpu':
    print('WARNING: no GPU detected. Runtime > Change runtime type > GPU (T4 is fine).')

## 2. Get the code


In [ ]:
import os

REPO_DIR = 'symmetric_parseval_conv'
if not os.path.isdir(REPO_DIR) and os.path.basename(os.getcwd()) != REPO_DIR:
    !git clone https://github.com/Theborna/symmetric_parseval_conv.git
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)
!git pull --ff-only
print('Working dir:', os.getcwd())

## 3. Install dependencies

Colab already ships `torch`, `torchvision` and `numpy`; we only add the lighter
packages the repo imports.


In [ ]:
!pip install -q tqdm tensorboard h5py einops scikit-image matplotlib piqa pytorch-ssim

# utils/utilities.py imports pytorch_ssim at module load; make sure it resolves.
try:
    import pytorch_ssim, piqa  # noqa: F401
    print('SSIM deps OK')
except Exception as e:
    print('installing pytorch_ssim from source:', e)
    !pip install -q git+https://github.com/Po-Hsun-Su/pytorch-ssim.git

## 4. Get your BSD500 data onto Colab

You need the pre-built HDF5 files `train.h5` and `test.h5` (the same ones you
train with locally). **Pick ONE option below** — each lands the files in
`/content/data/`, which the path cell at the end of this section expects. No
Google Drive account required.


### Option A — upload straight from your computer

Zero setup. Reliable for files up to a few hundred MB; slow/flaky for multi-GB
files (use Option B/C for those). A dialog will ask you to pick both files.


In [ ]:
import os
os.makedirs('/content/data', exist_ok=True)
from google.colab import files
print('Select train.h5 and test.h5 ...')
uploaded = files.upload()
for fn in uploaded:
    os.replace(fn, f'/content/data/{fn}')
print('saved:', os.listdir('/content/data'))

### Option B — download from a direct URL

Works with any direct link: Dropbox (append `?dl=1`), OneDrive, a personal /
university server, or a **GitHub Release asset** on your own repo (up to 2 GB
per file). Fill in the two URLs.


In [ ]:
import os
os.makedirs('/content/data', exist_ok=True)
TRAIN_URL = ''   # <-- direct link to train.h5
VAL_URL   = ''   # <-- direct link to test.h5
assert TRAIN_URL and VAL_URL, 'set TRAIN_URL and VAL_URL first'
!wget -q --show-progress -O /content/data/train.h5 "{TRAIN_URL}"
!wget -q --show-progress -O /content/data/test.h5  "{VAL_URL}"
print('saved:', os.listdir('/content/data'))

### Option C — Hugging Face Hub (durable, good for reruns)

Upload the two files once to a (private) dataset repo, e.g. with
`huggingface_hub.HfApi().upload_file(...)`, then pull them here. Best if you'll
rerun this notebook often.


In [ ]:
!pip install -q huggingface_hub
import os, shutil
from huggingface_hub import hf_hub_download  # , login

# login('hf_xxx')          # uncomment for a PRIVATE dataset repo
HF_REPO = 'your-username/bsd500'   # <-- your dataset repo id

os.makedirs('/content/data', exist_ok=True)
for fn in ['train.h5', 'test.h5']:
    p = hf_hub_download(repo_id=HF_REPO, filename=fn, repo_type='dataset')
    shutil.copy(p, f'/content/data/{fn}')
print('saved:', os.listdir('/content/data'))

### Option D — Google Drive

Only if you do have Drive access on this account.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# then set TRAIN_H5/VAL_H5 below to your Drive paths

### Set the paths and patch the configs (required)

Run this after whichever option above you used. It rewrites the data paths in
`config.json` and every `experiment_configs/*.json`.


In [ ]:
import glob, json

# Options A/B/C save here; for Drive (D) point these at your Drive paths.
TRAIN_H5 = '/content/data/train.h5'
VAL_H5   = '/content/data/test.h5'

assert os.path.exists(TRAIN_H5), f'train file not found: {TRAIN_H5}'
assert os.path.exists(VAL_H5),   f'val file not found: {VAL_H5}'

def patch_data_paths(train_h5, val_h5):
    for p in ['config.json'] + sorted(glob.glob('experiment_configs/*.json')):
        with open(p) as f:
            cfg = json.load(f)
        if 'training_options' in cfg:
            cfg['training_options']['train_data_file'] = train_h5
            cfg['training_options']['val_data_file'] = val_h5
            with open(p, 'w') as f:
                json.dump(cfg, f, indent=4)
            print('patched', p)

patch_data_paths(TRAIN_H5, VAL_H5)

## 5. Run the sweep

`EPOCHS` overrides every config (use `1` for a quick smoke test first). Leave
`CONFIGS` empty to run all four models, or set e.g. `--configs symmetric_mirror mirror`.
Each config's own `sigmas` (default `[5, 15, 25]`) are used.


In [ ]:
EPOCHS  = 1              # bump up (e.g. 10) for the real run
OUTPUT  = 'exps/paper'
CONFIGS = ''             # e.g. '--configs symmetric_mirror mirror'

!python experiments.py -d {device} --epochs {EPOCHS} -o {OUTPUT} {CONFIGS}

## 6. Results


In [ ]:
from IPython.display import Markdown, display
with open(os.path.join(OUTPUT, 'results.md')) as f:
    display(Markdown(f.read()))

In [ ]:
# Per-run view (best validation metrics + wall-clock minutes)
import pandas as pd
res = json.load(open(os.path.join(OUTPUT, 'results.json')))
rows = []
for name, d in res.items():
    for sigma, m in d.get('runs', {}).items():
        rows.append({'model': d.get('label', name), 'sigma': int(sigma),
                     'PSNR': round(m['best_psnr'], 2), 'SSIM': round(m['best_ssim'], 4),
                     'depth': m.get('depth'), 'width': m.get('width'),
                     'minutes': m.get('minutes')})
pd.DataFrame(rows).sort_values(['model', 'sigma']).reset_index(drop=True)

In [ ]:
# LaTeX table for the paper
print(open(os.path.join(OUTPUT, 'results.tex')).read())

## 7. (Optional) Save the results

Download the outputs so they survive the Colab session.


In [ ]:
# from google.colab import files
# files.download(os.path.join(OUTPUT, 'results.tex'))
# files.download(os.path.join(OUTPUT, 'results.json'))